In [1]:
import pandas as pd
import pickle

# Graph
with open('../data/user_graph.pkl', 'rb') as f:
    G = pickle.load(f)

# Features
user_features = pd.read_csv('../data/user_features.csv')
tweet_issue_labels = pd.read_csv('../data/tweet_issue_labels.csv')
issue_features = pd.read_csv('../data/issue_features.csv')

/var/folders/vr/vlqlwkjd2xl4vtv66h3t5nbw0000gn/T/ipykernel_13476/546592496.py:9: DtypeWarning: Columns (0: user) have mixed types. Specify dtype option on import or set low_memory=False.
  user_features = pd.read_csv('../data/user_features.csv')


### Build Node Index - map each graph node to unique integer ID

only using conversations, replies, hashtags - not the mentions

The problem with mention nodes is they have no overlap with tweet_issue_labels or user_features since those use numeric user IDs. 

Mention nodes carry less structural signal anyway since they're just screen names with no features.

In [2]:
import re

def clean_user(x):
    if isinstance(x, float):
        return str(int(x))
    if isinstance(x, str):
        x = x.strip()
        if x.startswith('{'):
            m = re.search(r"'id_str'\s*:\s*'(\d+)'", x)
            if m:
                return m.group(1)
            m = re.search(r"'id'\s*:\s*(\d+)", x)
            if m:
                return m.group(1)
        try:
            return str(int(float(x)))
        except (ValueError, TypeError):
            pass
    return str(x)


def is_user_node(n):
    s = str(n)
    return not s.startswith('hashtag_') and not s.startswith('conv_') and clean_user(n).isdigit()

# Build node index
# all_nodes_filtered = [n for n in G.nodes() if is_user_node(n) or str(n).startswith('hashtag_') or str(n).startswith('conv_')]
# node2idx = {clean_user(n) if is_user_node(n) else str(n): i for i, n in enumerate(all_nodes_filtered)}

# print(f"Total nodes after filtering: {len(node2idx)}")

all_nodes_filtered = [n for n in G.nodes() if is_user_node(n) or str(n).startswith('hashtag_') or str(n).startswith('conv_')]

# Deduplicate — keep first occurrence of each cleaned key
seen = {}
deduped_nodes = []
for n in all_nodes_filtered:
    key = clean_user(n) if is_user_node(n) else str(n)
    if key not in seen:
        seen[key] = True
        deduped_nodes.append(n)

# Build node2idx from deduplicated list
node2idx = {}
for i, n in enumerate(deduped_nodes):
    key = clean_user(n) if is_user_node(n) else str(n)
    node2idx[key] = i

print(f"Total filtered nodes (before dedup): {len(all_nodes_filtered)}")
print(f"Total filtered nodes (after dedup): {len(deduped_nodes)}")
print(f"Max index: {max(node2idx.values())}")
print(f"len(node2idx): {len(node2idx)}")

# Check node type breakdown
user_nodes = [n for n in node2idx if not n.startswith('hashtag_') and not n.startswith('conv_')]
hashtag_nodes = [n for n in node2idx if n.startswith('hashtag_')]
conv_nodes = [n for n in node2idx if n.startswith('conv_')]

print(f"User nodes: {len(user_nodes)}")
print(f"Hashtag nodes: {len(hashtag_nodes)}")
print(f"Conv nodes: {len(conv_nodes)}")

Total filtered nodes (before dedup): 365096
Total filtered nodes (after dedup): 284054
Max index: 284053
len(node2idx): 284054
User nodes: 161813
Hashtag nodes: 10844
Conv nodes: 111397


In [3]:
# what non-user, non-hashtag, non-conv nodes look like
other_nodes = [n for n in node2idx 
               if not str(n).startswith('hashtag_') 
               and not str(n).startswith('conv_')
               and not str(n).isdigit()]

print(f"Non-standard user nodes: {len(other_nodes)}")
print("Samples:", other_nodes[:10])

Non-standard user nodes: 0
Samples: []


### Edge Tensors for the 3 relation types (Conversations, Replies, Hashtags)

In [4]:
import torch

edge_index_dict = {'reply': [[], []], 'hashtag': [[], []], 'conversation': [[], []]}

skipped_edges = 0
for u, v, data in G.edges(data=True):
    edge_type = data.get('type')
    if edge_type not in edge_index_dict:
        continue  # skip mention edges

    # clean both nodes
    u_key = clean_user(u) if is_user_node(u) else str(u)
    v_key = clean_user(v) if is_user_node(v) else str(v)

    # skip if either node was filtered out (mention nodes)
    if u_key not in node2idx or v_key not in node2idx:
        skipped_edges += 1
        continue

    u_idx = node2idx[u_key]
    v_idx = node2idx[v_key]

    edge_index_dict[edge_type][0].append(u_idx)
    edge_index_dict[edge_type][1].append(v_idx)

# Convert to tensors
for etype in edge_index_dict:
    edge_index_dict[etype] = torch.tensor(edge_index_dict[etype], dtype=torch.long)
    print(f"{etype}: {edge_index_dict[etype].shape}")

print(f"Skipped edges (mention type): {skipped_edges}")


reply: torch.Size([2, 173323])
hashtag: torch.Size([2, 36475])
conversation: torch.Size([2, 241197])
Skipped edges (mention type): 0


### Node Feature Matrix

In [5]:
import numpy as np

num_nodes = len(node2idx)
feature_dim = 3  # degree, pagerank, betweenness

# Initialize with zeros
x = torch.zeros((num_nodes, feature_dim), dtype=torch.float)

# Fill in user node features from user_features dataframe
user_features['user'] = user_features['user'].astype(str)

matched = 0
skipped = 0
for _, row in user_features.iterrows():
    uid = str(row['user'])
    if uid in node2idx:
        idx = node2idx[uid]
        if idx >= num_nodes:  # safety check
            skipped += 1
            continue
        x[idx, 0] = row['degree']
        x[idx, 1] = row['pagerank']
        x[idx, 2] = row['betweenness']
        matched += 1
    else:
        skipped += 1

print(f"Total nodes: {num_nodes}")
print(f"User nodes matched with features: {matched}")
print(f"User nodes skipped (no features): {skipped}") # mention nodes that got filtered out
print(f"Feature matrix shape: {x.shape}")
print(f"Non-zero rows: {(x.sum(dim=1) != 0).sum().item()}")

Total nodes: 284054
User nodes matched with features: 284054
User nodes skipped (no features): 58249
Feature matrix shape: torch.Size([284054, 3])
Non-zero rows: 284054


### Issue Labels per User Node

In [6]:
# Assign dominant issue per user via majority vote
user_issue = (
    tweet_issue_labels.groupby('user')['issue_semantic']
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
    .rename(columns={'issue_semantic': 'dominant_issue'})
)

# Build issue to integer mapping
issues = sorted(user_issue['dominant_issue'].unique())
issue2idx = {issue: i for i, issue in enumerate(issues)}
idx2issue = {i: issue for issue, i in issue2idx.items()}

print(f"Number of issues: {len(issues)}")
print(issue2idx)

Number of issues: 22
{'Abortion': 0, 'Climate change': 1, 'Crime': 2, 'Democracy in the U.S.': 3, 'Distribution of income and wealth in the U.S.': 4, 'Education': 5, 'Energy policy': 6, 'Foreign affairs': 7, 'Gun policy': 8, 'Healthcare': 9, 'Immigration': 10, 'Race relations': 11, 'Relations with China': 12, 'Relations with Russia': 13, 'Situation in Middle East between Israelis and Palestinians': 14, 'Taxes': 15, 'Terrorism and national security': 16, 'The economy': 17, 'The federal budget deficit': 18, 'Trade with other nations': 19, 'Transgender rights': 20, 'Types of Supreme Court justices candidates would pick': 21}


In [7]:
### Building Label Tensor
# Initialize labels as -1 (unlabeled) for all nodes
y = torch.full((num_nodes,), -1, dtype=torch.long)

user_issue['user'] = user_issue['user'].astype(str)

labeled = 0
skipped = 0
for _, row in user_issue.iterrows():
    uid = str(row['user'])
    if uid in node2idx:
        idx = node2idx[uid]
        if idx >= num_nodes:
            skipped += 1
            continue
        y[idx] = issue2idx[row['dominant_issue']]
        labeled += 1
    else:
        skipped += 1

print(f"Labeled nodes: {labeled}")
print(f"Unlabeled nodes: {(y == -1).sum().item()}")
print(f"Skipped: {skipped}")

Labeled nodes: 129814
Unlabeled nodes: 154240
Skipped: 0


### Defining R-GCN Model

In [8]:
import subprocess
subprocess.run(['pip', 'install', 'torch-geometric', '--break-system-packages'])


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


CompletedProcess(args=['pip', 'install', 'torch-geometric', '--break-system-packages'], returncode=0)

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv

class RGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, embedding_dim, num_relations):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.conv2 = RGCNConv(hidden_channels, embedding_dim, num_relations)

    def forward(self, x, edge_index, edge_type):
        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index, edge_type)
        return x  # returns node embeddings, not class scores

# Combine all edge indices into one tensor with relation type labels
edge_index_combined = torch.cat([
    edge_index_dict['reply'],
    edge_index_dict['hashtag'],
    edge_index_dict['conversation']
], dim=1)

edge_type_combined = torch.cat([
    torch.zeros(edge_index_dict['reply'].shape[1], dtype=torch.long),
    torch.ones(edge_index_dict['hashtag'].shape[1], dtype=torch.long),
    torch.full((edge_index_dict['conversation'].shape[1],), 2, dtype=torch.long)
])

print(f"Combined edge index shape: {edge_index_combined.shape}")
print(f"Max node index in edges: {edge_index_combined.max().item()}")
print(f"Total nodes in feature matrix: {x.shape[0]}")

# Verify no out of bounds
assert edge_index_combined.max().item() < x.shape[0], "Still have out of bounds indices!"
print("All indices valid — safe to train")

# Initialize model
model = RGCN(
    in_channels=3,       # degree, pagerank, betweenness
    hidden_channels=64,
    embedding_dim=32,     
    num_relations=3      # reply, hashtag, conversation
)

print(model)

Combined edge index shape: torch.Size([2, 450995])
Max node index in edges: 284053
Total nodes in feature matrix: 284054
All indices valid — safe to train
RGCN(
  (conv1): RGCNConv(3, 64, num_relations=3)
  (conv2): RGCNConv(64, 32, num_relations=3)
)


In [10]:
print(max(node2idx.values()))
print(len(node2idx))

284053
284054


### Training Loop

In [11]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

losses = []

def train():
    model.train()
    optimizer.zero_grad()
    embeddings = model(x, edge_index_combined, edge_type_combined)

    # Positive pairs — connected nodes should be similar
    src, dst = edge_index_combined
    pos_score = (embeddings[src] * embeddings[dst]).sum(dim=1)
    
    # Negative pairs — random unconnected nodes
    neg_dst = torch.randint(0, num_nodes, (src.size(0),))
    neg_score = (embeddings[src] * embeddings[neg_dst]).sum(dim=1)

    loss = F.softplus(-pos_score).mean() + F.softplus(neg_score).mean()
    loss.backward()
    optimizer.step()
    return loss.item()

# Training
for epoch in range(1, 101):
    loss = train()
    losses.append(loss)
    print(f"Epoch {epoch:03d} | Loss: {loss:.4f}")

Epoch 001 | Loss: 9948.8838
Epoch 002 | Loss: 4694.2627
Epoch 003 | Loss: 3834.1348
Epoch 004 | Loss: 3219.2451
Epoch 005 | Loss: 3005.5742
Epoch 006 | Loss: 2661.0369
Epoch 007 | Loss: 2696.9756
Epoch 008 | Loss: 2369.0198
Epoch 009 | Loss: 1750.5583
Epoch 010 | Loss: 1643.1880
Epoch 011 | Loss: 1595.3398
Epoch 012 | Loss: 1485.6140
Epoch 013 | Loss: 1337.9534
Epoch 014 | Loss: 1310.0588
Epoch 015 | Loss: 1294.3979
Epoch 016 | Loss: 1046.2000
Epoch 017 | Loss: 1060.6488
Epoch 018 | Loss: 901.8032
Epoch 019 | Loss: 816.0928
Epoch 020 | Loss: 732.2427
Epoch 021 | Loss: 599.3770
Epoch 022 | Loss: 635.6862
Epoch 023 | Loss: 490.8958
Epoch 024 | Loss: 488.7739
Epoch 025 | Loss: 440.9915
Epoch 026 | Loss: 461.5092
Epoch 027 | Loss: 376.3261
Epoch 028 | Loss: 309.0540
Epoch 029 | Loss: 331.6175
Epoch 030 | Loss: 293.6844
Epoch 031 | Loss: 255.0643
Epoch 032 | Loss: 289.4406
Epoch 033 | Loss: 259.2541
Epoch 034 | Loss: 223.2177
Epoch 035 | Loss: 222.0202
Epoch 036 | Loss: 188.4781
Epoch 037 |

In [18]:
# Plot loss curve
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, 101)),
    y=losses,
    mode='lines+markers',
    line=dict(color='#378ADD', width=2),
    marker=dict(size=4),
    name='Training Loss'
))

fig.update_layout(
    title='R-GCN Training Loss',
    xaxis_title='Epoch',
    yaxis_title='Loss (log scale)',
    yaxis_type='log',
    template='plotly_white',
    height=400
)
fig.show()

### Aggregation Step

In [25]:
gallup = pd.read_csv('../data/gallup_ground_truth.csv')

# Rename the long issue column
gallup = gallup.rename(columns={gallup.columns[0]: 'issue'})
gallup['issue'] = gallup['issue'].str.strip()

# Create a weighted importance score and rank
gallup['importance_score'] = gallup['% Extremely important'] + 0.5 * gallup['% Very important']
gallup = gallup.sort_values('importance_score', ascending=False).reset_index(drop=True)
gallup['gallup_rank'] = range(1, len(gallup) + 1)

print(gallup[['issue', 'importance_score', 'gallup_rank']])
print(gallup.columns.tolist())

                                                issue  importance_score  \
0                                         The economy              71.0   
1                               Democracy in the U.S.              67.0   
2                     Terrorism and national security              64.0   
3   Types of Supreme Court justices candidates wou...              62.0   
4                                           Education              61.5   
5                                          Healthcare              58.0   
6                                         Immigration              56.5   
7                                          Gun policy              55.5   
8                                               Taxes              55.5   
9                                               Crime              55.0   
10                                           Abortion              51.5   
11                         The federal budget deficit              51.0   
12                       

In [26]:
twitter_issues = set(amplification_df['issue'])
gallup_issues = set(gallup['issue'])

print("In Twitter but not Gallup:")
print(twitter_issues - gallup_issues)

print("\nIn Gallup but not Twitter:")
print(gallup_issues - twitter_issues)

In Twitter but not Gallup:
set()

In Gallup but not Twitter:
set()


In [27]:
# After training, aggregate embeddings per issue
model.eval()
with torch.no_grad():
    embeddings = model(x, edge_index_combined, edge_type_combined)

# Map users to their issue and mean pool embeddings
user_issue['user'] = user_issue['user'].astype(str)

issue_embeddings = {}
for issue in issues:
    issue_users = user_issue[user_issue['dominant_issue'] == issue]['user'].tolist()
    indices = [node2idx[u] for u in issue_users if u in node2idx]
    if indices:
        issue_embeddings[issue] = embeddings[torch.tensor(indices)].mean(dim=0)

# Compute amplification score as L2 norm of issue embedding
amplification_scores = {
    issue: emb.norm().item() 
    for issue, emb in issue_embeddings.items()
}

# Rank issues by amplification score
amplification_df = pd.DataFrame({
    'issue': list(amplification_scores.keys()),
    'amplification_score': list(amplification_scores.values())
}).sort_values('amplification_score', ascending=False).reset_index(drop=True)

amplification_df['twitter_rank'] = range(1, len(amplification_df) + 1)
print(amplification_df)

                                                issue  amplification_score  \
0                                         The economy             3.459438   
1                                          Gun policy             3.378670   
2                                       Energy policy             2.477948   
3                               Democracy in the U.S.             2.336489   
4                                          Healthcare             2.224721   
5                          The federal budget deficit             2.204894   
6                                               Crime             1.957653   
7                                               Taxes             1.814404   
8                            Trade with other nations             1.731550   
9       Distribution of income and wealth in the U.S.             1.651431   
10                    Terrorism and national security             1.592321   
11                               Relations with China           

### Dissonance Comparison

In [69]:
# Merge with Gallup
from IPython.display import display
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

dissonance_df = amplification_df.merge(
    gallup[['issue', 'importance_score']],
    on='issue',
    how='left'
)

dissonance_df['dissonance'] = abs(
    dissonance_df['twitter_rank'] - dissonance_df['importance_score'].rank(ascending=False).astype(int)
)

display(dissonance_df.sort_values('dissonance', ascending=False).reset_index(drop=True))

,issue,amplification_score,twitter_rank,importance_score,dissonance
0,Education,0.989265,22,61.5,17
1,Energy policy,2.477948,3,45.0,13
2,Immigration,1.361371,20,56.5,13
3,Types of Supreme Court justices candidates would pick,1.457025,16,62.0,12
4,Trade with other nations,1.731550,9,42.5,9
5,Transgender rights,1.515297,14,28.0,8
6,Climate change,1.516504,13,35.5,8
7,Terrorism and national security,1.592321,11,64.0,8
8,Relations with China,1.541406,12,41.5,7
9,Gun policy,3.378670,2,55.5,6


Importance score high — voters consider this issue very important (e.g. The economy at 71.0).    
Importance score low — voters consider this issue less important (e.g. Transgender rights at 28.0).     
Twitter rank low (e.g. rank 1, 2, 3) — barely amplified on Twitter, peripheral in the network.     
Twitter rank high (e.g. rank 20, 21, 22) — highly amplified on Twitter, appears frequently and centrally in the network.    

Key findings:

- Education has the highest dissonance (17) — it's ranked 22nd on Twitter but voters rate it as 5th most important. Massively underamplified.
- Energy policy and Immigration (dissonance 13 each) — Energy policy is overamplified on Twitter (rank 3) relative to voter priority (rank 16). Immigration is underamplified (rank 20 on Twitter, rank 7 for voters).
- The economy, Healthcare, Taxes have near-zero dissonance — Twitter amplification aligns well with voter - priority for these.
- Gun policy is overamplified on Twitter (rank 2) vs voter priority (rank 8).

### Visualizations

In [29]:
dissonance_sorted = dissonance_df.sort_values('dissonance', ascending=True)

fig = go.Figure()

fig.add_trace(go.Bar(
    y=dissonance_sorted['issue'],
    x=dissonance_sorted['dissonance'],
    orientation='h',
    marker_color=dissonance_sorted['dissonance'],
    marker_colorscale='RdYlGn_r',
    marker_showscale=True,
    marker_colorbar=dict(title='Dissonance')
))

fig.update_layout(
    title='Dissonance: Twitter Amplification vs Voter Priority (Gallup)',
    xaxis_title='Dissonance Score (rank difference)',
    template='plotly_white',
    height=700
)
fig.show()

Plot 1 (bar chart) shows the dissonance score — which is the absolute rank difference between Twitter amplification rank and Gallup voter priority rank. Education has a dissonance of 17 meaning its Twitter rank (22nd) and Gallup rank (5th) differ by 17 positions. It's a direct measure of how misaligned each issue is.

Key takeaway:
Biggest mismatches: Education, Immigration, Energy policy → these are the most “out of sync.”
Small mismatches: Economy, Taxes, Healthcare → these are more aligned between Twitter and public priorities.

In [66]:
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=dissonance_df['importance_score'].rank(ascending=False).astype(int),
    y=dissonance_df['twitter_rank'],
    mode='markers+text',
    text=dissonance_df['issue'].str[:70],  # increase from 20 to 30
    textposition='top center',
    textfont=dict(size=8),
    name='Issues',  # fixes "trace 0"
    marker=dict(
        size=dissonance_df['dissonance'] * 2 + 5,
        color=dissonance_df['dissonance'],
        colorscale='RdYlGn_r',
        # showscale=True,
        # colorbar=dict(
        #     title='Dissonance',
        #     x=1.15,
        #     thickness=15,
        # )
    )
))

fig2.add_trace(go.Scatter(
    x=list(range(1, 23)),
    y=list(range(1, 23)),
    mode='lines',
    line=dict(color='grey', dash='dash'),
    name='Perfect alignment'
))

fig2.update_layout(
    title='Twitter Amplification Rank vs Gallup Voter Priority Rank',
    xaxis_title='Gallup Rank (voter priority)',
    yaxis_title='Twitter Rank (amplification)',
    template='plotly_white',
    height=600,
    # width=1500,  # wider to fit labels
    # margin=dict(r=200),  # more right margin for colorbar
    legend=dict(x=0.95, y=0.99999, xanchor='left', yanchor='top')
)
fig2.show()

Plot 2 (scatter) shows the raw ranks themselves — x-axis is Gallup rank, y-axis is Twitter rank. The diagonal is perfect alignment. Points above the diagonal are overamplified on Twitter relative to voter importance (Education, Immigration, Types of Supreme Court). Points below the diagonal are underamplified on Twitter (Energy policy, Gun policy). The bubble size and color both encode the dissonance score from plot 1.

Key takeaways:
Over-amplified on Twitter:
Issues like education, immigration, abortion, race relations get more attention online than their priority among voters.
Under-amplified on Twitter:
Issues like economy, healthcare, energy policy, gun policy matter more to voters but get less attention on Twitter.
Fairly aligned:
Some issues (e.g., foreign affairs, taxes) are closer to the line.
Color/size = mismatch (dissonance):
Bigger, redder bubbles = larger gap between Twitter and real-world priorities.
Bottom line:
Twitter does not reflect what most voters actually prioritize—some issues are overhyped, while others that matter more to people are relatively ignored.

### Spearman Rank Correlation

In [70]:
from scipy.stats import spearmanr

gallup_ranks = dissonance_df['importance_score'].rank(ascending=False).astype(int)
twitter_ranks = dissonance_df['twitter_rank']

corr, pvalue = spearmanr(gallup_ranks, twitter_ranks)

print(f"Spearman Rank Correlation: {corr:.4f}")
print(f"P-value: {pvalue:.4f}")

Spearman Rank Correlation: 0.2791
P-value: 0.2084


- Correlation of 0.28 — very weak positive correlation. Twitter amplification barely aligns with what voters actually prioritize. The two rankings are largely independent of each other.
- P-value of 0.21 — not statistically significant (above the 0.05 threshold). This means we cannot confidently rule out that the weak correlation is just due to chance with this sample of 22 issues.

Interpretation: Twitter is not amplifying issues in proportion to voter importance. The near-zero correlation suggests the platform's network dynamics — reply chains, hashtag adoption, user centrality — are driven by factors other than genuine public priority.
The caveat is that with only 22 data points (issues), the statistical power is limited, which is why the p-value is not significant.

### Save Results

In [71]:
dissonance_df.to_csv('../data/rgcn_dissonance.csv', index=False)